## **Working with Geospatial Data – Session 6**

This session introduces advanced geospatial analysis using:

* H3 hexagonal spatial indexing 🛑
* MinMaxScaler for normalization 📊
* Scikit-learn clustering 📈
* GeoPandas for vector data processing 🗺

In [1]:
pip install scikit-learn geopandas h3pandas h3~=3.0 -q


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import geopandas as gpd
import h3pandas
from shapely.geometry import Point, Polygon
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import pandas as pd

In [3]:
schools = gpd.read_file("../data/nyc/nyc_school_points/SchoolPoints_APS_2024_08_28.shp")
subways = gpd.read_file("../data/nyc/nyc_subway_entrances/nyc_subway_entrances.shp")
bike_paths = gpd.read_file("../data/nyc/nyc_bike_routes_20241223.geojson")
neighborhoods = gpd.read_file("https://raw.githubusercontent.com/HodgesWardElliott/custom-nyc-neighborhoods/refs/heads/master/custom-pedia-cities-nyc-Mar2018.geojson")
parks = gpd.read_file("../data/nyc/Parks Properties_20241223.geojson")

In [4]:
schools = schools.to_crs("EPSG:3857")
subways = subways.to_crs("EPSG:3857")
bike_paths = bike_paths.to_crs("EPSG:3857")
neighborhoods = neighborhoods.to_crs("EPSG:3857")
parks = parks.to_crs("EPSG:3857")

# Analyze neighborhoods

In [5]:
# Analysis for each neighborhood
def analyze_neighborhood(neighborhood_geometry):
    # Count features intersecting the neighborhood boundary
    num_schools = schools[schools.geometry.intersects(neighborhood_geometry)].shape[0]
    num_subways = subways[subways.geometry.intersects(neighborhood_geometry)].shape[0]
    bike_path_length = bike_paths[bike_paths.geometry.intersects(neighborhood_geometry)].length.sum()
    park_area = parks[parks.geometry.intersects(neighborhood_geometry)].area.sum()

    return num_schools, num_subways, bike_path_length, park_area

In [6]:
# Apply analysis to each neighborhood
neighborhoods[['num_schools', 'num_subways', 'bike_path_length', 'park_area']] = neighborhoods.geometry.apply(
    lambda geom: pd.Series(analyze_neighborhood(geom))
)

In [7]:
neighborhoods.head(3)

,neighborhood,boroughCode,borough,X.id,geometry,num_schools,num_subways,bike_path_length,park_area
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8220788.214 4993431.406, -8220479.3...",12.0,5.0,12972.782695,5.191057e+05
1,Alley Pond Park,4,Queens,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8209070.244 4973902.938, -8209112.6...",0.0,0.0,5886.886135,7.365924e+06
2,Arden Heights,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8256547.374 4947814.753, -8256546.8...",1.0,0.0,0.000000,9.533262e+06


In [8]:
neighborhoods.head(3)

,neighborhood,boroughCode,borough,X.id,geometry,num_schools,num_subways,bike_path_length,park_area
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8220788.214 4993431.406, -8220479.3...",12.0,5.0,12972.782695,5.191057e+05
1,Alley Pond Park,4,Queens,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8209070.244 4973902.938, -8209112.6...",0.0,0.0,5886.886135,7.365924e+06
2,Arden Heights,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8256547.374 4947814.753, -8256546.8...",1.0,0.0,0.000000,9.533262e+06


In [9]:
# Normalize results (0 to 1 scale)
scaler = MinMaxScaler()
columns_to_normalize = ['num_schools', 'num_subways', 'bike_path_length', 'park_area']
neighborhoods[columns_to_normalize] = scaler.fit_transform(neighborhoods[columns_to_normalize])

In [10]:
# Aggregate results using touching neighborhoods
def aggregate_touching_neighborhoods(neighborhood_index):
    current_geometry = neighborhoods.loc[neighborhood_index, 'geometry']
    touching_indices = neighborhoods[neighborhoods.geometry.touches(current_geometry)].index

    if not touching_indices.empty:
        neighbor_values = neighborhoods.loc[touching_indices, columns_to_normalize].mean()
    else:
        neighbor_values = neighborhoods.loc[neighborhood_index, columns_to_normalize]

    return neighbor_values

In [11]:
# Apply aggregation to each neighborhood
neighborhoods[columns_to_normalize] = neighborhoods.index.to_series().apply(
    lambda idx: aggregate_touching_neighborhoods(idx)
)


In [12]:
neighborhoods[columns_to_normalize] = scaler.fit_transform(neighborhoods[columns_to_normalize])

In [13]:
neighborhoods.head(3)

,neighborhood,boroughCode,borough,X.id,geometry,num_schools,num_subways,bike_path_length,park_area
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8220788.214 4993431.406, -8220479.3...",0.066667,0.036585,0.009243,0.025121
1,Alley Pond Park,4,Queens,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8209070.244 4973902.938, -8209112.6...",0.185185,0.000000,0.203034,0.322221
2,Arden Heights,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8256547.374 4947814.753, -8256546.8...",0.040741,0.016260,0.079246,0.210843


In [14]:
neighborhoods['index_score'] = neighborhoods['num_schools'] + neighborhoods['num_subways'] + neighborhoods['bike_path_length'] + neighborhoods['park_area'] 

In [15]:
import leafmap

m = leafmap.Map()
m.add_data(
    neighborhoods, column="index_score", scheme="Quantiles", cmap="Blues", legend_title="Index"
)
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

In [16]:
neighborhoods.to_file("neighborhood_access_index.geojson", driver="GeoJSON")

# Set up the H3 Grid

In [17]:
neighborhoods = neighborhoods.to_crs('EPSG:4326')

In [18]:
resolution = 9  # Adjust resolution as needed
gdf_h3 = neighborhoods.h3.polyfill(resolution)

In [19]:
gdf_h3.head()

,neighborhood,boroughCode,borough,X.id,geometry,num_schools,num_subways,bike_path_length,park_area,index_score,category,color,h3_polyfill
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702...",0.066667,0.036585,0.009243,0.025121,0.137616,1,#f7fbff,"[892a100102fffff, 892a100106bffff, 892a1001067..."
1,Alley Pond Park,4,Queens,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.74333 40.73888, -73.74371 40.739...",0.185185,0.000000,0.203034,0.322221,0.710440,4,#2171b5,"[892a10056a7ffff, 892a100509bffff, 892a10050a7..."
2,Arden Heights,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.16983 40.56108, -74.16982 40.561...",0.040741,0.016260,0.079246,0.210843,0.347090,2,#c6dbef,"[892a1060e93ffff, 892a10608c3ffff, 892a1060ed7..."
3,Arlington,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.15975 40.64142, -74.15998 40.641...",0.000000,0.000000,0.000000,0.012868,0.012868,1,#f7fbff,[892a106267bffff]
4,Arrochar,5,Staten Island,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-74.06078 40.59319, -74.06079 40.593...",0.016667,0.015244,0.080113,0.136984,0.249007,2,#c6dbef,"[892a107533bffff, 892a1075317ffff, 892a107531b..."


In [20]:
gdf_h3 = neighborhoods.h3.polyfill(resolution, explode=True)
gdf_h3.head()

,neighborhood,boroughCode,borough,X.id,geometry,num_schools,num_subways,bike_path_length,park_area,index_score,category,color,h3_polyfill
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702...",0.066667,0.036585,0.009243,0.025121,0.137616,1,#f7fbff,892a100102fffff
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702...",0.066667,0.036585,0.009243,0.025121,0.137616,1,#f7fbff,892a100106bffff
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702...",0.066667,0.036585,0.009243,0.025121,0.137616,1,#f7fbff,892a1001067ffff
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702...",0.066667,0.036585,0.009243,0.025121,0.137616,1,#f7fbff,892a1001027ffff
0,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702...",0.066667,0.036585,0.009243,0.025121,0.137616,1,#f7fbff,892a1001383ffff


In [21]:
gdf_h3 = gdf_h3[gdf_h3['h3_polyfill'].isnull() == False].set_index('h3_polyfill')
gdf_h3.index.name = None
gdf_h3

,neighborhood,boroughCode,borough,X.id,geometry,num_schools,num_subways,bike_path_length,park_area,index_score,category,color
892a100102fffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702...",0.066667,0.036585,0.009243,0.025121,0.137616,1,#f7fbff
892a100106bffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702...",0.066667,0.036585,0.009243,0.025121,0.137616,1,#f7fbff
892a1001067ffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702...",0.066667,0.036585,0.009243,0.025121,0.137616,1,#f7fbff
892a1001027ffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702...",0.066667,0.036585,0.009243,0.025121,0.137616,1,#f7fbff
892a1001383ffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-73.8486 40.87167, -73.84582 40.8702...",0.066667,0.036585,0.009243,0.025121,0.137616,1,#f7fbff
...,...,...,...,...,...,...,...,...,...,...,...,...
892a1008ddbffff,Harlem,1,Manhattan,None,"POLYGON ((-73.93457 40.82815, -73.93442 40.827...",0.507407,0.278455,0.584689,0.141232,1.511784,5,#08306b
892a100abcfffff,Harlem,1,Manhattan,None,"POLYGON ((-73.93457 40.82815, -73.93442 40.827...",0.507407,0.278455,0.584689,0.141232,1.511784,5,#08306b
892a1008d43ffff,Harlem,1,Manhattan,None,"POLYGON ((-73.93457 40.82815, -73.93442 40.827...",0.507407,0.278455,0.584689,0.141232,1.511784,5,#08306b
892a100ab43ffff,Harlem,1,Manhattan,None,"POLYGON ((-73.93457 40.82815, -73.93442 40.827...",0.507407,0.278455,0.584689,0.141232,1.511784,5,#08306b


In [22]:
gdf_h3 = gdf_h3.h3.h3_to_geo_boundary()

In [23]:
pip install folium matplotlib mapclassify -q


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
gdf_h3.explore()

In [25]:
gdf_h3_proj = gdf_h3.to_crs('EPSG:3857')

In [26]:
# Analysis for each hex cell
def analyze_access(hex_geometry):
    # Buffer hex geometry
    buffer_1600m = hex_geometry.buffer(1600)
    buffer_800m = hex_geometry.buffer(800)

    # Count features within buffers
    num_schools = schools[schools.geometry.intersects(buffer_1600m)].shape[0]
    num_subways = subways[subways.geometry.intersects(buffer_1600m)].shape[0]
    bike_path_length = bike_paths[bike_paths.geometry.intersects(buffer_1600m)].length.sum()
    park_area = parks[parks.geometry.intersects(buffer_800m)].area.sum()

    return num_schools, num_subways, bike_path_length, park_area

In [27]:
gdf_h3_proj = gdf_h3.to_crs('EPSG:3857')

In [28]:
gdf_h3_proj[['num_schools', 'num_subways', 'bike_path_length', 'park_area']] = gdf_h3_proj.geometry.apply(
    lambda hex_geom: pd.Series(analyze_access(hex_geom))
)


In [29]:
gdf_h3_proj.head()

,neighborhood,boroughCode,borough,X.id,geometry,num_schools,num_subways,bike_path_length,park_area,index_score,category,color
892a100102fffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8221754.701 4992626.011, -8221989.6...",20.0,12.0,32762.831164,5.066436e+05,0.137616,1,#f7fbff
892a100106bffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8223207.83 4993355.679, -8223442.82...",28.0,12.0,32755.030721,5.656904e+06,0.137616,1,#f7fbff
892a1001067ffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8222481.225 4992990.858, -8222716.2...",26.0,13.0,33296.033540,5.656789e+06,0.137616,1,#f7fbff
892a1001027ffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8221277.541 4992642.309, -8221512.5...",19.0,6.0,27148.171444,5.066436e+05,0.137616,1,#f7fbff
892a1001383ffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8222459.658 4992212.209, -8222694.6...",16.0,11.0,27713.531366,5.100501e+06,0.137616,1,#f7fbff


In [30]:
gdf_h3_proj['h3_index'] = gdf_h3_proj.index

# Run the normalization analysis

In [31]:
import h3

In [32]:
# Normalize results
scaler = MinMaxScaler()
normalized_columns = ['num_schools', 'num_subways', 'bike_path_length', 'park_area']
gdf_h3_proj[normalized_columns] = scaler.fit_transform(gdf_h3_proj[normalized_columns])

# Aggregate results using neighboring cells
def aggregate_neighbors(h3_index):
    neighbors = h3.k_ring(h3_index, 2)  # 2-k ring
    neighbor_values = gdf_h3_proj[gdf_h3_proj['h3_index'].isin(neighbors)][normalized_columns].mean()
    return neighbor_values

gdf_h3_proj[normalized_columns] = gdf_h3_proj['h3_index'].apply(
    lambda h3_index: aggregate_neighbors(h3_index)
)

# # Final normalized analysis
gdf_h3_proj[normalized_columns] = scaler.fit_transform(gdf_h3_proj[normalized_columns])

# Save or visualize the results
gdf_h3_proj.to_file("access_index.geojson", driver="GeoJSON")

In [33]:
gdf_h3_proj.head(3)

,neighborhood,boroughCode,borough,X.id,geometry,num_schools,num_subways,bike_path_length,park_area,index_score,category,color,h3_index
892a100102fffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8221754.701 4992626.011, -8221989.6...",0.234554,0.045739,0.287617,0.093918,0.137616,1,#f7fbff,892a100102fffff
892a100106bffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8223207.83 4993355.679, -8223442.82...",0.274027,0.084944,0.305051,0.285792,0.137616,1,#f7fbff,892a100106bffff
892a1001067ffff,Allerton,2,Bronx,http://nyc.pediacities.com/Resource/Neighborho...,"POLYGON ((-8222481.225 4992990.858, -8222716.2...",0.258581,0.060441,0.311282,0.189247,0.137616,1,#f7fbff,892a1001067ffff


# Create the total score

In [34]:
gdf_h3_proj['index_score'] = gdf_h3_proj['num_schools'] + gdf_h3_proj['num_subways'] + gdf_h3_proj['bike_path_length'] + gdf_h3_proj['park_area'] 

In [35]:
import leafmap

In [36]:
gdf_h3_map = gdf_h3_proj.to_crs('EPSG:4326')

In [37]:
m = leafmap.Map()
m.add_data(
    gdf_h3_map, column="index_score", scheme="Quantiles", cmap="Blues", legend_title="Index"
)
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…